# 04 — Non-Pest Relevance Gate

## AI-Based Insect Pest Detection and Management Recommendation System

This notebook trains a **relevance gate** for the final pest-detection system.

The gate performs binary classification:

- `0` → Not a supported pest image
- `1` → Supported pest image

Only images accepted by this gate are passed to the final 10-class MobileNetV2 pest classifier.

**Dataset used:** 40 non-pest images + 40 supported pest images  
**Image size:** 128 × 128  
**Model:** MobileNetV2 transfer learning  
**Saved model:** `gate_model.keras`


## 1. Configuration and paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

DATASET_PATH = Path(r"C:\Users\jeeva\Downloads\ip102_v1.1")
NEGATIVE_PATH = Path(
    r"C:\Users\jeeva\OneDrive\Desktop\AI pest finder\not_pest\not_pest_images_40"
)
GATE_MODEL_PATH = Path(
    r"C:\Users\jeeva\OneDrive\Desktop\AI pest finder\gate_model.keras"
)

GATE_IMG_SIZE = (128, 128)
GATE_BATCH_SIZE = 8
RANDOM_STATE = 42

print("IP102 dataset:", DATASET_PATH)
print("Non-pest images:", NEGATIVE_PATH)
print("Gate model:", GATE_MODEL_PATH)

IP102 dataset: C:\Users\jeeva\Downloads\ip102_v1.1
Non-pest images: C:\Users\jeeva\OneDrive\Desktop\AI pest finder\not_pest\not_pest_images_40
Gate model: C:\Users\jeeva\OneDrive\Desktop\AI pest finder\gate_model.keras


## 2. Load the supported pest images

In [3]:
selected_classes = {
    0: "rice leaf roller",
    3: "asiatic rice borer",
    7: "brown plant hopper",
    11: "rice leafhopper",
    18: "black cutworm",
    22: "corn borer",
    23: "army worm",
    24: "aphids",
    54: "Thrips",
    71: "Trialeurodes vaporariorum"
}

def read_ip102_split(file_path):
    records = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            parts = line.split()

            records.append({
                "image": parts[0],
                "class_id": int(parts[1])
            })

    return pd.DataFrame(records)

train_df = read_ip102_split(DATASET_PATH / "train.txt")

train_selected = train_df[
    train_df["class_id"].isin(selected_classes.keys())
].copy()

positive_df = train_selected[["image"]].copy()
positive_df["binary_label"] = 1

print("Supported pest images available:", len(positive_df))

Supported pest images available: 7612


## 3. Load the non-pest images and create a balanced gate dataset

In [5]:
image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

negative_images = [
    p for p in NEGATIVE_PATH.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]

negative_df = pd.DataFrame({
    "image": [str(p) for p in negative_images],
    "binary_label": 0
})

if len(negative_df) != 40:
    print(f"Warning: expected 40 non-pest images, found {len(negative_df)}")

# Use 40 supported pest images so both classes are balanced.
positive_sample = positive_df.sample(
    n=40,
    random_state=RANDOM_STATE
)

balanced_df = pd.concat(
    [positive_sample, negative_df],
    ignore_index=True
)

def fix_gate_image_path(image_path):
    image_path = Path(str(image_path))

    if image_path.is_absolute():
        return str(image_path)

    return str(DATASET_PATH / "images" / image_path.name)

balanced_df["image"] = balanced_df["image"].apply(fix_gate_image_path)

balanced_df = balanced_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("Balanced gate dataset:", len(balanced_df))
print("\nClass distribution:")
print(balanced_df["binary_label"].value_counts())

Balanced gate dataset: 80

Class distribution:
binary_label
1    40
0    40
Name: count, dtype: int64


## 4. Split into training and validation sets

In [7]:
gate_train_df, gate_val_df = train_test_split(
    balanced_df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=balanced_df["binary_label"]
)

print("Gate training images:", len(gate_train_df))
print("Gate validation images:", len(gate_val_df))

print("\nTraining distribution:")
print(gate_train_df["binary_label"].value_counts())

print("\nValidation distribution:")
print(gate_val_df["binary_label"].value_counts())

Gate training images: 60
Gate validation images: 20

Training distribution:
binary_label
0    30
1    30
Name: count, dtype: int64

Validation distribution:
binary_label
0    10
1    10
Name: count, dtype: int64


## 5. Create TensorFlow datasets

In [9]:
def load_gate_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(
        image,
        [GATE_IMG_SIZE[0], GATE_IMG_SIZE[1]]
    )
    image = tf.cast(image, tf.float32) / 255.0

    return image, label


def create_gate_dataset(df, shuffle=False):
    image_paths = df["image"].values
    labels = df["binary_label"].values.astype(np.float32)

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    dataset = dataset.map(
        load_gate_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=RANDOM_STATE
        )

    dataset = dataset.batch(GATE_BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


gate_train_dataset = create_gate_dataset(
    gate_train_df,
    shuffle=True
)

gate_val_dataset = create_gate_dataset(
    gate_val_df,
    shuffle=False
)

print(
    "Training batches:",
    tf.data.experimental.cardinality(gate_train_dataset).numpy()
)
print(
    "Validation batches:",
    tf.data.experimental.cardinality(gate_val_dataset).numpy()
)

Training batches: 8
Validation batches: 3


## 6. Build the MobileNetV2 relevance gate

In [11]:
from tensorflow.keras import layers, models

gate_base = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights="imagenet"
)

gate_base.trainable = False

gate_model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    # Convert [0, 1] pixels to the [-1, 1] range expected by MobileNetV2.
    layers.Rescaling(scale=2.0, offset=-1.0),

    gate_base,

    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation="relu"),

    layers.Dropout(0.4),

    # 0 = not supported, 1 = supported pest
    layers.Dense(1, activation="sigmoid")
])

gate_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gate_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)                │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_128 (Functional)    │ (None, 4, 4, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │          81,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 7. Train the relevance gate

In [13]:
gate_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

gate_history = gate_model.fit(
    gate_train_dataset,
    validation_data=gate_val_dataset,
    epochs=15,
    callbacks=[gate_early_stopping]
)

Epoch 1/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 55s 3s/step - accuracy: 0.5333 - loss: 0.7460 - val_accuracy: 0.5500 - val_loss: 0.6923
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 710ms/step - accuracy: 0.7000 - loss: 0.6120 - val_accuracy: 0.7000 - val_loss: 0.6205
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 449ms/step - accuracy: 0.7333 - loss: 0.5432 - val_accuracy: 0.7000 - val_loss: 0.5576
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 302ms/step - accuracy: 0.7167 - loss: 0.5363 - val_accuracy: 0.7000 - val_loss: 0.5087
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 527ms/step - accuracy: 0.8333 - loss: 0.3611 - val_accuracy: 0.7500 - val_loss: 0.4670
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 387ms/step - accuracy: 0.9000 - loss: 0.3198 - val_accuracy: 0.7500 - val_loss: 0.4317
Epoch 7/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 368ms/step - accuracy: 0.8667 - loss: 0.3450 - val_accuracy: 0.8000 - val_loss: 0.4044
Epoch 8/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 271ms/step - accuracy: 0.8500 - loss: 0.3315 - val_accuracy: 0.8000 - val_loss: 0

## 8. Evaluate the gate

In [15]:
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in gate_val_dataset:
    probabilities = gate_model.predict(images, verbose=0).flatten()
    predicted_labels = (probabilities >= 0.5).astype(int)

    y_true.extend(labels.numpy().astype(int))
    y_pred.extend(predicted_labels)

cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Not Supported Pest",
            "Supported Pest"
        ]
    )
)

Confusion Matrix:
[[ 9  1]
 [ 0 10]]

Classification Report:
                    precision    recall  f1-score   support

Not Supported Pest       1.00      0.90      0.95        10
    Supported Pest       0.91      1.00      0.95        10

          accuracy                           0.95        20
         macro avg       0.95      0.95      0.95        20
      weighted avg       0.95      0.95      0.95        20



## 9. Save and verify the final gate model

In [17]:
GATE_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

gate_model.save(GATE_MODEL_PATH)

print("Gate model saved successfully:")
print(GATE_MODEL_PATH)

print("\nFile exists:", GATE_MODEL_PATH.exists())
if GATE_MODEL_PATH.exists():
    print(
        "File size (MB):",
        round(GATE_MODEL_PATH.stat().st_size / (1024 * 1024), 2)
    )

loaded_gate_model = tf.keras.models.load_model(GATE_MODEL_PATH)
print("\nGate model loaded successfully.")

Gate model saved successfully:
C:\Users\jeeva\OneDrive\Desktop\AI pest finder\gate_model.keras

File exists: True
File size (MB): 10.12

Gate model loaded successfully.


## 10. Test the gate

The following cells test the saved gate with:

1. a non-pest/family image → expected rejection
2. a known IP102 pest image → expected acceptance

The gate threshold is **0.50**.

In [19]:
from PIL import Image

def gate_predict(image_path, model=loaded_gate_model):
    image = Image.open(image_path).convert("RGB")
    image = image.resize(GATE_IMG_SIZE)

    image_array = np.array(
        image,
        dtype=np.float32
    ) / 255.0

    image_batch = np.expand_dims(
        image_array,
        axis=0
    )

    probability = float(
        model.predict(
            image_batch,
            verbose=0
        )[0][0]
    )

    return probability


family_image_path = Path(
    r"C:\Users\jeeva\OneDrive\Desktop\AI pest finder\family.jpg"
)

if family_image_path.exists():
    probability = gate_predict(family_image_path)

    print("Non-pest test image:", family_image_path)
    print("Gate probability:", round(probability, 4))
    print("Gate probability (%):", round(probability * 100, 2))
    print(
        "Result:",
        "SUPPORTED PEST IMAGE" if probability >= 0.5
        else "NOT A SUPPORTED PEST IMAGE"
    )
else:
    print("Optional non-pest test image not found:")
    print(family_image_path)

Non-pest test image: C:\Users\jeeva\OneDrive\Desktop\AI pest finder\family.jpg
Gate probability: 0.2533
Gate probability (%): 25.33
Result: NOT A SUPPORTED PEST IMAGE


In [20]:
pest_image_path = Path(
    r"C:\Users\jeeva\Downloads\ip102_v1.1\images\01865.jpg"
)

if pest_image_path.exists():
    probability = gate_predict(pest_image_path)

    print("Known pest test image:", pest_image_path)
    print("Gate probability:", round(probability, 4))
    print("Gate probability (%):", round(probability * 100, 2))
    print(
        "Result:",
        "SUPPORTED PEST IMAGE" if probability >= 0.5
        else "NOT A SUPPORTED PEST IMAGE"
    )
else:
    print("Known pest test image not found:")
    print(pest_image_path)

Known pest test image: C:\Users\jeeva\Downloads\ip102_v1.1\images\01865.jpg
Gate probability: 0.6145
Gate probability (%): 61.45
Result: SUPPORTED PEST IMAGE


## Final gate result

The relevance gate is used before the 10-class pest classifier:

**Uploaded image → Relevance Gate →**
- **Not supported:** reject image
- **Supported pest:** send image to the 10-class MobileNetV2 classifier

This prevents the 10-class classifier from being used directly on clearly unrelated images.